# Build and validate the Neo4j network

Creates the Buyer-Supplier graph, loads it into Neo4j, validates totals, and exports network metrics for downstream analysis.

Run this notebook from the `notebooks/` directory after completing the preceding numbered stage. Generated files are written to the documented project directories.


# Build the final network in Neo4j


In [ ]:
from pathlib import Path
import os
import math
import pandas as pd
import numpy as np

import certifi
from dotenv import load_dotenv
from neo4j import GraphDatabase
from IPython.display import display

os.environ.setdefault("SSL_CERT_FILE", certifi.where())

load_dotenv(Path(".env"))
load_dotenv(Path("notebooks/.env"))

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)


In [ ]:
# Path configuration

ruta_procesados = Path("../data/processed")
ruta_neo4j = Path("../data/neo4j")

ruta_neo4j.mkdir(parents=True, exist_ok=True)

archivo_procedimientos = ruta_procesados / "procedures_2025.csv"
archivo_oferentes = ruta_procesados / "tender_participation_2025.csv"
archivo_adjudicaciones = ruta_procesados / "awards_2025.csv"

for archivo in [
    archivo_procedimientos,
    archivo_oferentes,
    archivo_adjudicaciones,
]:
    if not archivo.exists():
        raise FileNotFoundError(
            f"No se encontró {archivo}. "
            "Ejecutar primero el notebook 02_clean_and_integrate_data."
        )

print("Archivos procesados encontrados correctamente.")


In [ ]:
# Load the processed datasets

procedimientos_df = pd.read_csv(
    archivo_procedimientos,
    dtype={
        "ocid": "string",
        "buyer_id": "string",
        "buyer_ruc": "string",
    }
)

oferentes_df = pd.read_csv(
    archivo_oferentes,
    dtype={
        "ocid": "string",
        "tenderer_id": "string",
    }
)

adjudicaciones_df = pd.read_csv(
    archivo_adjudicaciones,
    dtype={
        "ocid": "string",
        "award_id": "string",
        "buyer_id": "string",
        "supplier_id": "string",
        "supplier_ruc": "string",
        "cpc_id": "string",
        "cpc_5": "string",
    }
)

print("Procedimientos:", procedimientos_df.shape)
print("Analyze bidder participation:", oferentes_df.shape)
print("Adjudicaciones:", adjudicaciones_df.shape)


In [ ]:
# Validate inputs before building the graph

columnas_procedimientos_requeridas = {
    "ocid",
    "buyer_id",
    "buyer_name",
}

columnas_adjudicaciones_requeridas = {
    "ocid",
    "award_id",
    "buyer_id",
    "supplier_id",
    "award_amount",
    "cpc_5",
    "award_date_local",
}

faltantes_proc = columnas_procedimientos_requeridas - set(procedimientos_df.columns)
faltantes_awards = columnas_adjudicaciones_requeridas - set(adjudicaciones_df.columns)

if faltantes_proc:
    raise ValueError(
        "Faltan columnas requeridas en procedimientos_df: "
        + ", ".join(sorted(faltantes_proc))
    )

if faltantes_awards:
    raise ValueError(
        "Faltan columnas requeridas en adjudicaciones_df: "
        + ", ".join(sorted(faltantes_awards))
    )

controles = {
    "procedimientos": len(procedimientos_df),
    "ocid_unicos": procedimientos_df["ocid"].nunique(),
    "ocid_nulos": int(procedimientos_df["ocid"].isna().sum()),
    "buyer_id_nulos_procedimientos": int(procedimientos_df["buyer_id"].isna().sum()),
    "adjudicaciones": len(adjudicaciones_df),
    "supplier_id_nulos": int(adjudicaciones_df["supplier_id"].isna().sum()),
    "award_amount_nulos": int(adjudicaciones_df["award_amount"].isna().sum()),
    "cpc_5_nulos": int(adjudicaciones_df["cpc_5"].isna().sum()),
    "award_date_local_nulos": int(adjudicaciones_df["award_date_local"].isna().sum()),
    "buyers_unicos": procedimientos_df["buyer_id"].nunique(),
    "suppliers_unicos": adjudicaciones_df["supplier_id"].nunique(),
}

display(pd.Series(controles, name="resultado").to_frame())

if controles["procedimientos"] != controles["ocid_unicos"]:
    raise ValueError("El OCID no es único en la tabla de procedimientos.")

duplicados_award = adjudicaciones_df.duplicated(
    subset=["ocid", "award_id", "supplier_id"]
).sum()

print(f"Duplicados OCID + award_id + supplier_id: {duplicados_award:,}")

if duplicados_award > 0:
    raise ValueError(
        "Se detectaron adjudicaciones duplicadas. "
        "Revisar la fase de limpieza antes de construir la red."
    )


In [ ]:
comprador_por_ocid = (
    procedimientos_df[["ocid", "buyer_id"]]
    .drop_duplicates("ocid")
    .rename(columns={"buyer_id": "buyer_id_procedimiento"})
)

validacion_buyer = adjudicaciones_df[
    ["ocid", "buyer_id"]
].merge(
    comprador_por_ocid,
    on="ocid",
    how="left"
)

diferencias_buyer = validacion_buyer[
    validacion_buyer["buyer_id"] != validacion_buyer["buyer_id_procedimiento"]
]

print(
    "Adjudicaciones con buyer_id diferente al procedimiento:",
    len(diferencias_buyer)
)

if len(diferencias_buyer) > 0:
    display(diferencias_buyer.head(20))
    raise ValueError(
        "Existen inconsistencias comprador–procedimiento. "
        "No se debe continuar con la carga."
    )


In [ ]:
def nombre_mas_frecuente(serie):
    valores = (
        serie
        .dropna()
        .astype("string")
        .str.strip()
    )
    valores = valores[
        ~valores.str.lower().isin(["", "null", "none", "nan"])
    ]
    if valores.empty:
        return None
    return valores.value_counts().index[0]


# -----------------------------
# NODOS BUYER
# -----------------------------

columnas_buyer = ["buyer_id", "buyer_name"]

for col in [
    "buyer_name_normalized",
    "buyer_ruc",
    "identificador_comprador_atipico",
]:
    if col in procedimientos_df.columns:
        columnas_buyer.append(col)

buyers_df = procedimientos_df[columnas_buyer].copy()

buyers_df = (
    buyers_df
    .groupby("buyer_id", as_index=False, dropna=False)
    .agg({
        col: nombre_mas_frecuente if buyers_df[col].dtype == "object" or str(buyers_df[col].dtype).startswith("string")
        else "first"
        for col in buyers_df.columns
        if col != "buyer_id"
    })
)

buyers_df = buyers_df[buyers_df["buyer_id"].notna()].copy()


# -----------------------------
# NODOS SUPPLIER
# -----------------------------

nombre_supplier_col = (
    "nombre_proveedor_limpio"
    if "nombre_proveedor_limpio" in adjudicaciones_df.columns
    else "supplier_name"
)

columnas_supplier = ["supplier_id", nombre_supplier_col]

for col in [
    "supplier_name_normalized",
    "supplier_ruc",
    "supplier_ruc_validado",
    "identificador_proveedor_atipico",
]:
    if col in adjudicaciones_df.columns:
        columnas_supplier.append(col)

suppliers_df = adjudicaciones_df[columnas_supplier].copy()

agg_supplier = {}
for col in suppliers_df.columns:
    if col == "supplier_id":
        continue
    if suppliers_df[col].dtype == "object" or str(suppliers_df[col].dtype).startswith("string"):
        agg_supplier[col] = nombre_mas_frecuente
    else:
        agg_supplier[col] = "first"

suppliers_df = (
    suppliers_df
    .groupby("supplier_id", as_index=False, dropna=False)
    .agg(agg_supplier)
)

suppliers_df = suppliers_df[suppliers_df["supplier_id"].notna()].copy()

if nombre_supplier_col != "supplier_name":
    suppliers_df = suppliers_df.rename(
        columns={nombre_supplier_col: "supplier_name"}
    )


# -----------------------------
# NODOS PROCEDURE
# -----------------------------

columnas_procedure = [
    "ocid",
    "tender_id",
    "tender_status",
    "number_of_tenderers",
]

for col in [
    "tender_value",
    "tender_start_date",
    "release_date",
    "tiene_adjudicacion",
    "registro_atipico_calidad",
]:
    if col in procedimientos_df.columns:
        columnas_procedure.append(col)

columnas_procedure = [
    col for col in columnas_procedure
    if col in procedimientos_df.columns
]

procedures_graph_df = (
    procedimientos_df[columnas_procedure]
    .drop_duplicates("ocid")
    .copy()
)


# -----------------------------
# NODOS CPC
# -----------------------------

cpc_name_col = (
    "cpc_description"
    if "cpc_description" in adjudicaciones_df.columns
    else None
)

columnas_cpc = ["cpc_5"]
if cpc_name_col:
    columnas_cpc.append(cpc_name_col)

cpc_df = adjudicaciones_df[columnas_cpc].copy()
cpc_df = cpc_df[cpc_df["cpc_5"].notna()].copy()

if cpc_name_col:
    cpc_df = (
        cpc_df
        .groupby("cpc_5", as_index=False)
        .agg({cpc_name_col: nombre_mas_frecuente})
    )
else:
    cpc_df = cpc_df.drop_duplicates("cpc_5")


print("Buyers:", len(buyers_df))
print("Suppliers:", len(suppliers_df))
print("Procedures:", len(procedures_graph_df))
print("CPC5:", len(cpc_df))


In [ ]:
# Create procedure-supplier relationships

# Buyer -> Procedure
buyer_procedure_df = (
    procedimientos_df[
        ["buyer_id", "ocid"]
    ]
    .dropna()
    .drop_duplicates()
    .copy()
)


columnas_award_rel = [
    "ocid",
    "award_id",
    "supplier_id",
    "award_amount",
    "award_date_local",
    "cpc_5",
]

for col in [
    "award_currency",
    "award_amount_source",
]:
    if col in adjudicaciones_df.columns:
        columnas_award_rel.append(col)

procedure_supplier_df = (
    adjudicaciones_df[columnas_award_rel]
    .dropna(subset=["ocid", "supplier_id"])
    .copy()
)

procedure_supplier_df["award_amount"] = pd.to_numeric(
    procedure_supplier_df["award_amount"],
    errors="coerce"
)


# Procedure -> CPC
procedure_cpc_df = (
    adjudicaciones_df[
        ["ocid", "cpc_5"]
    ]
    .dropna()
    .drop_duplicates()
    .copy()
)

print("Relaciones Buyer -> Procedure:", len(buyer_procedure_df))
print("Registros Procedure -> Supplier:", len(procedure_supplier_df))
print("Relaciones Procedure -> CPC:", len(procedure_cpc_df))


In [ ]:
base_relacion = adjudicaciones_df[
    [
        "buyer_id",
        "supplier_id",
        "ocid",
        "award_amount",
        "award_date_local",
    ]
].copy()

base_relacion["award_amount"] = pd.to_numeric(
    base_relacion["award_amount"],
    errors="coerce"
)

base_relacion["award_date_dt"] = pd.to_datetime(
    base_relacion["award_date_local"],
    errors="coerce"
)

buyer_supplier_df = (
    base_relacion
    .dropna(subset=["buyer_id", "supplier_id"])
    .groupby(
        ["buyer_id", "supplier_id"],
        as_index=False
    )
    .agg(
        frequency=("ocid", "nunique"),
        total_amount=("award_amount", "sum"),
        average_amount=("award_amount", "mean"),
        first_award_date=("award_date_dt", "min"),
        last_award_date=("award_date_dt", "max"),
    )
)

buyer_supplier_df["log10_total_amount"] = np.where(
    buyer_supplier_df["total_amount"] > 0,
    np.log10(buyer_supplier_df["total_amount"]),
    np.nan
)

print(
    "Relaciones únicas comprador-proveedor:",
    len(buyer_supplier_df)
)

display(
    buyer_supplier_df
    .sort_values("total_amount", ascending=False)
    .head(10)
)


In [ ]:
# Validate the constructed network

buyers_ids = set(buyers_df["buyer_id"])
supplier_ids = set(suppliers_df["supplier_id"])
procedure_ids = set(procedures_graph_df["ocid"])
cpc_ids = set(cpc_df["cpc_5"])

errores = {
    "buyer_procedure_sin_buyer": int(
        (~buyer_procedure_df["buyer_id"].isin(buyers_ids)).sum()
    ),
    "buyer_procedure_sin_procedure": int(
        (~buyer_procedure_df["ocid"].isin(procedure_ids)).sum()
    ),
    "award_sin_procedure": int(
        (~procedure_supplier_df["ocid"].isin(procedure_ids)).sum()
    ),
    "award_sin_supplier": int(
        (~procedure_supplier_df["supplier_id"].isin(supplier_ids)).sum()
    ),
    "buyer_supplier_sin_buyer": int(
        (~buyer_supplier_df["buyer_id"].isin(buyers_ids)).sum()
    ),
    "buyer_supplier_sin_supplier": int(
        (~buyer_supplier_df["supplier_id"].isin(supplier_ids)).sum()
    ),
}

display(pd.Series(errores, name="errores").to_frame())

if any(valor > 0 for valor in errores.values()):
    raise ValueError(
        "La preparación del grafo contiene relaciones con nodos inexistentes."
    )

frecuencia_esperada = (
    adjudicaciones_df[
        ["buyer_id", "supplier_id", "ocid"]
    ]
    .dropna()
    .drop_duplicates()
    .shape[0]
)

frecuencia_grafo = int(buyer_supplier_df["frequency"].sum())

print(f"Frecuencia esperada: {frecuencia_esperada:,}")
print(f"Suma de frequency en CONTRACTS_WITH: {frecuencia_grafo:,}")

if frecuencia_esperada != frecuencia_grafo:
    raise ValueError(
        "La frecuencia agregada comprador-proveedor no coincide "
        "con los procedimientos distintos observados."
    )

print("\nValidaciones estructurales superadas.")


In [ ]:
# Save graph import tables

buyers_df.to_csv(
    ruta_neo4j / "buyers_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

suppliers_df.to_csv(
    ruta_neo4j / "suppliers_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

procedures_graph_df.to_csv(
    ruta_neo4j / "procedures_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

cpc_df.to_csv(
    ruta_neo4j / "cpc5_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

buyer_procedure_df.to_csv(
    ruta_neo4j / "buyer_procedure_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

procedure_supplier_df.to_csv(
    ruta_neo4j / "procedure_supplier_awards_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

procedure_cpc_df.to_csv(
    ruta_neo4j / "procedure_cpc5_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

buyer_supplier_df.to_csv(
    ruta_neo4j / "buyer_supplier_aggregated_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Archivos de carga guardados en: {ruta_neo4j.resolve()}")


In [ ]:
# Connect to Neo4j

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

faltantes_config = [
    nombre
    for nombre, valor in {
        "NEO4J_URI": NEO4J_URI,
        "NEO4J_USERNAME": NEO4J_USERNAME,
        "NEO4J_PASSWORD": NEO4J_PASSWORD,
    }.items()
    if not valor
]

if faltantes_config:
    raise EnvironmentError(
        "Faltan variables de entorno para Neo4j: "
        + ", ".join(faltantes_config)
    )

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("Conexión con Neo4j verificada.")
print("Base de datos:", NEO4J_DATABASE)


In [ ]:
# Batch-loading helpers

def valor_python(valor):
    """Convierte valores pandas/numpy a tipos serializables por Neo4j."""
    if valor is None:
        return None

    if isinstance(valor, pd.Timestamp):
        if pd.isna(valor):
            return None
        return valor.isoformat()

    if isinstance(valor, np.generic):
        valor = valor.item()

    try:
        if pd.isna(valor):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(valor, (np.bool_, bool)):
        return bool(valor)

    return valor


def dataframe_a_registros(df):
    return [
        {
            columna: valor_python(valor)
            for columna, valor in fila.items()
        }
        for fila in df.to_dict(orient="records")
    ]


def ejecutar_query(query, **parameters):
    with driver.session(database=NEO4J_DATABASE) as session:
        return session.run(query, **parameters).consume()


def cargar_por_lotes(query, df, batch_size=1000, descripcion="registros"):
    registros = dataframe_a_registros(df)
    total = len(registros)

    for inicio in range(0, total, batch_size):
        lote = registros[inicio:inicio + batch_size]

        with driver.session(database=NEO4J_DATABASE) as session:
            session.execute_write(
                lambda tx: tx.run(
                    query,
                    rows=lote
                ).consume()
            )

        fin = min(inicio + batch_size, total)
        print(
            f"\r{descripcion}: {fin:,}/{total:,}",
            end=""
        )

    print()


In [ ]:
RESET_CAPSTONE_GRAPH = True # RECONSTRUYE DESDE CERO SI TRUE!!

if RESET_CAPSTONE_GRAPH:
    ejecutar_query(
        """
        MATCH (n)
        WHERE n.dataset = 'SIE_2025'
        DETACH DELETE n
        """
    )
    print("Grafo SIE_2025 eliminado.")
else:
    print("RESET_CAPSTONE_GRAPH=False. No se eliminó ningún dato.")


In [ ]:
constraints = [
    """
    CREATE CONSTRAINT buyer_id_unique IF NOT EXISTS
    FOR (b:Buyer)
    REQUIRE b.buyer_id IS UNIQUE
    """,
    """
    CREATE CONSTRAINT supplier_id_unique IF NOT EXISTS
    FOR (s:Supplier)
    REQUIRE s.supplier_id IS UNIQUE
    """,
    """
    CREATE CONSTRAINT procedure_ocid_unique IF NOT EXISTS
    FOR (p:Procedure)
    REQUIRE p.ocid IS UNIQUE
    """,
    """
    CREATE CONSTRAINT cpc5_unique IF NOT EXISTS
    FOR (c:CPC)
    REQUIRE c.cpc_5 IS UNIQUE
    """,
]

for query in constraints:
    ejecutar_query(query)

print("Restricciones creadas o verificadas correctamente.")


In [ ]:
# Cargamos nodos buyer

query_buyers = """
UNWIND $rows AS row
MERGE (b:Buyer {buyer_id: row.buyer_id})
SET b.dataset = 'SIE_2025',
    b.name = row.buyer_name,
    b.name_normalized = row.buyer_name_normalized,
    b.ruc = row.buyer_ruc,
    b.atypical_id = row.identificador_comprador_atipico
"""

buyers_load_df = buyers_df.copy()

for col in [
    "buyer_name",
    "buyer_name_normalized",
    "buyer_ruc",
    "identificador_comprador_atipico",
]:
    if col not in buyers_load_df.columns:
        buyers_load_df[col] = None

cargar_por_lotes(
    query_buyers,
    buyers_load_df[
        [
            "buyer_id",
            "buyer_name",
            "buyer_name_normalized",
            "buyer_ruc",
            "identificador_comprador_atipico",
        ]
    ],
    descripcion="Buyers"
)


In [ ]:
# Cargamos nodos supplier

query_suppliers = """
UNWIND $rows AS row
MERGE (s:Supplier {supplier_id: row.supplier_id})
SET s.dataset = 'SIE_2025',
    s.name = row.supplier_name,
    s.name_normalized = row.supplier_name_normalized,
    s.ruc = row.supplier_ruc,
    s.ruc_validated = row.supplier_ruc_validado,
    s.atypical_id = row.identificador_proveedor_atipico
"""

suppliers_load_df = suppliers_df.copy()

for col in [
    "supplier_name",
    "supplier_name_normalized",
    "supplier_ruc",
    "supplier_ruc_validado",
    "identificador_proveedor_atipico",
]:
    if col not in suppliers_load_df.columns:
        suppliers_load_df[col] = None

cargar_por_lotes(
    query_suppliers,
    suppliers_load_df[
        [
            "supplier_id",
            "supplier_name",
            "supplier_name_normalized",
            "supplier_ruc",
            "supplier_ruc_validado",
            "identificador_proveedor_atipico",
        ]
    ],
    descripcion="Suppliers"
)


In [ ]:
# Cargamos nodos Procedure

query_procedures = """
UNWIND $rows AS row
MERGE (p:Procedure {ocid: row.ocid})
SET p.dataset = 'SIE_2025',
    p.tender_id = row.tender_id,
    p.tender_status = row.tender_status,
    p.number_of_tenderers = row.number_of_tenderers,
    p.tender_value = row.tender_value,
    p.tender_start_date = row.tender_start_date,
    p.release_date = row.release_date,
    p.has_award = row.tiene_adjudicacion,
    p.quality_outlier = row.registro_atipico_calidad
"""

procedures_load_df = procedures_graph_df.copy()

for col in [
    "tender_id",
    "tender_status",
    "number_of_tenderers",
    "tender_value",
    "tender_start_date",
    "release_date",
    "tiene_adjudicacion",
    "registro_atipico_calidad",
]:
    if col not in procedures_load_df.columns:
        procedures_load_df[col] = None

cargar_por_lotes(
    query_procedures,
    procedures_load_df[
        [
            "ocid",
            "tender_id",
            "tender_status",
            "number_of_tenderers",
            "tender_value",
            "tender_start_date",
            "release_date",
            "tiene_adjudicacion",
            "registro_atipico_calidad",
        ]
    ],
    descripcion="Procedures"
)


In [ ]:
# Cargamos nodos CPC

query_cpc = """
UNWIND $rows AS row
MERGE (c:CPC {cpc_5: row.cpc_5})
SET c.dataset = 'SIE_2025',
    c.description = row.cpc_description
"""

cpc_load_df = cpc_df.copy()

if "cpc_description" not in cpc_load_df.columns:
    cpc_load_df["cpc_description"] = None

cargar_por_lotes(
    query_cpc,
    cpc_load_df[
        ["cpc_5", "cpc_description"]
    ],
    descripcion="CPC5"
)


In [ ]:
# Creamos relaciones

query_buyer_procedure = """
UNWIND $rows AS row
MATCH (b:Buyer {buyer_id: row.buyer_id})
MATCH (p:Procedure {ocid: row.ocid})
MERGE (b)-[r:INITIATED]->(p)
SET r.dataset = 'SIE_2025'
"""

cargar_por_lotes(
    query_buyer_procedure,
    buyer_procedure_df,
    descripcion="Buyer -> Procedure"
)


In [ ]:
query_procedure_supplier = """
UNWIND $rows AS row
MATCH (p:Procedure {ocid: row.ocid})
MATCH (s:Supplier {supplier_id: row.supplier_id})
MERGE (p)-[r:AWARDED_TO {award_id: row.award_id}]->(s)
SET r.dataset = 'SIE_2025',
    r.amount = row.award_amount,
    r.award_date = row.award_date_local,
    r.cpc_5 = row.cpc_5,
    r.currency = row.award_currency,
    r.amount_source = row.award_amount_source
"""

procedure_supplier_load_df = procedure_supplier_df.copy()

for col in [
    "award_currency",
    "award_amount_source",
]:
    if col not in procedure_supplier_load_df.columns:
        procedure_supplier_load_df[col] = None

cargar_por_lotes(
    query_procedure_supplier,
    procedure_supplier_load_df[
        [
            "ocid",
            "award_id",
            "supplier_id",
            "award_amount",
            "award_date_local",
            "cpc_5",
            "award_currency",
            "award_amount_source",
        ]
    ],
    descripcion="Procedure -> Supplier"
)


In [ ]:
query_procedure_cpc = """
UNWIND $rows AS row
MATCH (p:Procedure {ocid: row.ocid})
MATCH (c:CPC {cpc_5: row.cpc_5})
MERGE (p)-[r:CLASSIFIED_AS]->(c)
SET r.dataset = 'SIE_2025'
"""

cargar_por_lotes(
    query_procedure_cpc,
    procedure_cpc_df,
    descripcion="Procedure -> CPC"
)


In [ ]:
query_buyer_supplier = """
UNWIND $rows AS row
MATCH (b:Buyer {buyer_id: row.buyer_id})
MATCH (s:Supplier {supplier_id: row.supplier_id})
MERGE (b)-[r:CONTRACTS_WITH]->(s)
SET r.dataset = 'SIE_2025',
    r.frequency = row.frequency,
    r.total_amount = row.total_amount,
    r.average_amount = row.average_amount,
    r.log10_total_amount = row.log10_total_amount,
    r.first_award_date = row.first_award_date,
    r.last_award_date = row.last_award_date
"""

cargar_por_lotes(
    query_buyer_supplier,
    buyer_supplier_df,
    descripcion="Buyer -> Supplier agregado"
)


In [ ]:
# Post-load validation

query_conteos = """
MATCH (n)
WHERE n.dataset = 'SIE_2025'
WITH
    count(CASE WHEN n:Buyer THEN 1 END) AS buyers,
    count(CASE WHEN n:Supplier THEN 1 END) AS suppliers,
    count(CASE WHEN n:Procedure THEN 1 END) AS procedures,
    count(CASE WHEN n:CPC THEN 1 END) AS cpc
RETURN buyers, suppliers, procedures, cpc
"""

with driver.session(database=NEO4J_DATABASE) as session:
    conteos_nodos = session.run(query_conteos).single().data()

query_relaciones = """
MATCH ()-[r]->()
WHERE r.dataset = 'SIE_2025'
RETURN
    type(r) AS tipo,
    count(r) AS cantidad
ORDER BY tipo
"""

with driver.session(database=NEO4J_DATABASE) as session:
    conteos_rel = [
        record.data()
        for record in session.run(query_relaciones)
    ]

print("Nodos en Neo4j:")
display(pd.Series(conteos_nodos, name="neo4j").to_frame())

print("\nRelaciones en Neo4j:")
display(pd.DataFrame(conteos_rel))


In [ ]:
esperado_nodos = {
    "buyers": len(buyers_df),
    "suppliers": len(suppliers_df),
    "procedures": len(procedures_graph_df),
    "cpc": len(cpc_df),
}

esperado_relaciones = {
    "INITIATED": len(buyer_procedure_df),
    "AWARDED_TO": len(procedure_supplier_df),
    "CLASSIFIED_AS": len(procedure_cpc_df),
    "CONTRACTS_WITH": len(buyer_supplier_df),
}

neo4j_rel_dict = {
    fila["tipo"]: fila["cantidad"]
    for fila in conteos_rel
}

validacion_final = []

for tipo, esperado in esperado_nodos.items():
    obtenido = conteos_nodos.get(tipo, 0)
    validacion_final.append({
        "elemento": tipo,
        "esperado": esperado,
        "neo4j": obtenido,
        "coincide": esperado == obtenido,
    })

for tipo, esperado in esperado_relaciones.items():
    obtenido = neo4j_rel_dict.get(tipo, 0)
    validacion_final.append({
        "elemento": tipo,
        "esperado": esperado,
        "neo4j": obtenido,
        "coincide": esperado == obtenido,
    })

validacion_final_df = pd.DataFrame(validacion_final)

display(validacion_final_df)

if not validacion_final_df["coincide"].all():
    raise ValueError(
        "Los conteos de Neo4j no coinciden con los datos preparados. "
        "Revisar la carga antes de continuar."
    )

print("Validación final superada: la red coincide con las bases preparadas.")


In [ ]:
# Queries

query_top_monto = """
MATCH (b:Buyer)-[r:CONTRACTS_WITH]->(s:Supplier)
WHERE r.dataset = 'SIE_2025'
RETURN
    b.name AS buyer,
    s.name AS supplier,
    r.frequency AS frequency,
    r.total_amount AS total_amount
ORDER BY total_amount DESC
LIMIT 10
"""

with driver.session(database=NEO4J_DATABASE) as session:
    top_monto = pd.DataFrame(
        [record.data() for record in session.run(query_top_monto)]
    )

display(top_monto)


In [ ]:
query_top_frecuencia = """
MATCH (b:Buyer)-[r:CONTRACTS_WITH]->(s:Supplier)
WHERE r.dataset = 'SIE_2025'
RETURN
    b.name AS buyer,
    s.name AS supplier,
    r.frequency AS frequency,
    r.total_amount AS total_amount
ORDER BY frequency DESC, total_amount DESC
LIMIT 10
"""

with driver.session(database=NEO4J_DATABASE) as session:
    top_frecuencia = pd.DataFrame(
        [record.data() for record in session.run(query_top_frecuencia)]
    )

display(top_frecuencia)


In [ ]:
query_trazabilidad = """
MATCH (b:Buyer)-[r:CONTRACTS_WITH]->(s:Supplier)
WHERE r.dataset = 'SIE_2025'
WITH b, s, r
ORDER BY r.frequency DESC, r.total_amount DESC
LIMIT 1

MATCH (b)-[:INITIATED]->(p:Procedure)-[a:AWARDED_TO]->(s)

OPTIONAL MATCH (p)-[:CLASSIFIED_AS]->(c:CPC)

RETURN
    b.name AS buyer,
    s.name AS supplier,
    p.ocid AS ocid,
    a.award_id AS award_id,
    a.amount AS amount,
    a.award_date AS award_date,
    c.cpc_5 AS cpc_5,
    c.description AS cpc_description
ORDER BY award_date
"""

with driver.session(database=NEO4J_DATABASE) as session:
    trazabilidad = pd.DataFrame(
        [record.data() for record in session.run(query_trazabilidad)]
    )

display(trazabilidad)


In [ ]:
resumen_red = pd.DataFrame(
    {
        "Métrica": [
            "Entidades contratantes",
            "Proveedores adjudicados",
            "Procedimientos",
            "Categorías CPC5",
            "Relaciones buyer-procedure",
            "Adjudicaciones procedure-supplier",
            "Relaciones buyer-supplier únicas",
            "Monto total representado",
            "Frecuencia total representada",
        ],
        "Valor": [
            len(buyers_df),
            len(suppliers_df),
            len(procedures_graph_df),
            len(cpc_df),
            len(buyer_procedure_df),
            len(procedure_supplier_df),
            len(buyer_supplier_df),
            buyer_supplier_df["total_amount"].sum(),
            buyer_supplier_df["frequency"].sum(),
        ],
    }
)

display(resumen_red)


# Final validation and network characterization

This step documents and validates the corresponding stage of the analytical workflow. See the project report for the methodological rationale and interpretation.


## Final QA for temporal coverage, values, and awards


In [ ]:
tender_start = pd.to_datetime(
    procedimientos_df["tender_start_date"],
    errors="coerce"
) if "tender_start_date" in procedimientos_df.columns else pd.Series(dtype="datetime64[ns]")

award_date = pd.to_datetime(
    adjudicaciones_df["award_date_local"],
    errors="coerce"
)

if not tender_start.empty:
    resumen_tender_year = (
        tender_start.dt.year
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis("anio_inicio_procedimiento")
        .reset_index(name="cantidad")
    )
    print("Año de inicio de los procedimientos:")
    display(resumen_tender_year)

resumen_award_year = (
    award_date.dt.year
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("anio_adjudicacion")
    .reset_index(name="cantidad")
)

print("Año de adjudicación:")
display(resumen_award_year)

print("Primera adjudicación:", award_date.min())
print("Última adjudicación:", award_date.max())

if not tender_start.empty:
    anios_inicio_validos = set(tender_start.dropna().dt.year.unique())
    if anios_inicio_validos != {2025}:
        raise ValueError(
            f"Se encontraron procedimientos con año de inicio distinto de 2025: "
            f"{sorted(anios_inicio_validos)}"
        )

print("Validación temporal del universo superada.")


In [ ]:
if "tender_start_date" in procedimientos_df.columns:
    fechas_proc = (
        procedimientos_df[["ocid", "tender_start_date"]]
        .drop_duplicates("ocid")
        .copy()
    )

    fechas_proc["tender_start_dt"] = pd.to_datetime(
        fechas_proc["tender_start_date"],
        errors="coerce",
        utc=True
    ).dt.tz_convert("America/Guayaquil")

    validacion_fechas = adjudicaciones_df[
        ["ocid", "award_date_local"]
    ].merge(
        fechas_proc[["ocid", "tender_start_dt"]],
        on="ocid",
        how="left"
    )

    award_dt = pd.to_datetime(
        validacion_fechas["award_date_local"],
        errors="coerce"
    )

    if award_dt.dt.tz is None:
        award_dt = award_dt.dt.tz_localize("America/Guayaquil")
    else:
        award_dt = award_dt.dt.tz_convert("America/Guayaquil")

    validacion_fechas["award_date_dt"] = award_dt

    awards_antes_inicio = validacion_fechas[
        validacion_fechas["award_date_dt"].notna()
        & validacion_fechas["tender_start_dt"].notna()
        & (
            validacion_fechas["award_date_dt"]
            < validacion_fechas["tender_start_dt"]
        )
    ]

    print(
        "Adjudicaciones anteriores al inicio del procedimiento:",
        len(awards_antes_inicio)
    )

    if len(awards_antes_inicio) > 0:
        display(awards_antes_inicio.head(20))
        raise ValueError(
            "Existen adjudicaciones anteriores al inicio del procedimiento."
        )


In [ ]:
award_amounts = pd.to_numeric(
    adjudicaciones_df["award_amount"],
    errors="coerce"
)

monto_adjudicaciones = float(award_amounts.sum())
monto_contracts_with = float(
    pd.to_numeric(
        buyer_supplier_df["total_amount"],
        errors="coerce"
    ).sum()
)

diferencia_monto = monto_contracts_with - monto_adjudicaciones

qa_montos = pd.DataFrame({
    "metrica": [
        "Monto total en adjudicaciones",
        "Monto total en CONTRACTS_WITH",
        "Diferencia",
        "Montos nulos",
        "Montos <= 0",
    ],
    "valor": [
        monto_adjudicaciones,
        monto_contracts_with,
        diferencia_monto,
        int(award_amounts.isna().sum()),
        int((award_amounts <= 0).sum()),
    ],
})

display(qa_montos)

if not np.isclose(
    monto_adjudicaciones,
    monto_contracts_with,
    rtol=1e-10,
    atol=0.01
):
    raise ValueError(
        "El monto total de CONTRACTS_WITH no coincide con "
        "el monto total de las adjudicaciones."
    )

print("QA de montos superado.")


In [ ]:
percentiles_monto = award_amounts.quantile(
    [0.50, 0.95, 0.99, 0.999]
)

resumen_montos = pd.DataFrame({
    "metrica": [
        "Cantidad de montos",
        "Mínimo",
        "Mediana (P50)",
        "P95",
        "P99",
        "P99.9",
        "Máximo",
        "Media",
    ],
    "valor": [
        int(award_amounts.notna().sum()),
        award_amounts.min(),
        percentiles_monto.loc[0.50],
        percentiles_monto.loc[0.95],
        percentiles_monto.loc[0.99],
        percentiles_monto.loc[0.999],
        award_amounts.max(),
        award_amounts.mean(),
    ],
})

display(resumen_montos)


In [ ]:
ocids_con_award = set(
    adjudicaciones_df["ocid"]
    .dropna()
    .unique()
)

procedimientos_con_award = procedimientos_df[
    procedimientos_df["ocid"].isin(ocids_con_award)
].copy()

procedimientos_sin_award = procedimientos_df[
    ~procedimientos_df["ocid"].isin(ocids_con_award)
].copy()

cobertura_awards = pd.DataFrame({
    "categoria": [
        "Procedimientos totales",
        "Con al menos una adjudicación",
        "Sin adjudicación representada",
    ],
    "cantidad": [
        len(procedimientos_df),
        len(procedimientos_con_award),
        len(procedimientos_sin_award),
    ],
})

cobertura_awards["porcentaje"] = (
    cobertura_awards["cantidad"]
    / len(procedimientos_df)
    * 100
)

display(cobertura_awards)

if "tender_status" in procedimientos_sin_award.columns:
    print("Estado de los procedimientos sin adjudicación:")
    display(
        procedimientos_sin_award["tender_status"]
        .value_counts(dropna=False)
        .rename_axis("tender_status")
        .reset_index(name="cantidad")
    )


In [ ]:
n_buyers = len(buyers_df)
n_suppliers = len(suppliers_df)
n_edges = len(buyer_supplier_df)
n_nodes = n_buyers + n_suppliers

densidad_bipartita = (
    n_edges / (n_buyers * n_suppliers)
    if n_buyers > 0 and n_suppliers > 0
    else np.nan
)

estadisticas_globales = pd.DataFrame({
    "metrica": [
        "Entidades contratantes",
        "Proveedores adjudicados",
        "Nodos bipartitos",
        "Relaciones Buyer-Supplier únicas",
        "Densidad bipartita",
        "Frecuencia promedio por relación",
        "Monto promedio por relación",
    ],
    "valor": [
        n_buyers,
        n_suppliers,
        n_nodes,
        n_edges,
        densidad_bipartita,
        buyer_supplier_df["frequency"].mean(),
        buyer_supplier_df["total_amount"].mean(),
    ],
})

display(estadisticas_globales)


In [ ]:
GDS_DISPONIBLE = False
GDS_VERSION = None
GDS_ENVIRONMENT = None

try:
    with driver.session(database=NEO4J_DATABASE) as session:
        registro = session.run(
            """
            CALL gds.graph.list()
            YIELD graphName
            RETURN count(graphName) AS graph_count
            """
        ).single()

    GDS_DISPONIBLE = True
    GDS_ENVIRONMENT = "Aura Graph Analytics / GDS"
    GDS_VERSION = "versionless"

    print("Neo4j GDS disponible.")
    print("Entorno:", GDS_ENVIRONMENT)
    print("Versión:", GDS_VERSION)
    print(
        "Proyecciones GDS existentes:",
        registro["graph_count"],
    )

except Exception as exc:
    raise RuntimeError(
        "No fue posible acceder a las capacidades de Neo4j Graph Data Science. "
        "El Notebook 05 requiere GDS para calcular betweenness "
        "y generar actor_network_metrics_2025.csv completo."
    ) from exc


In [ ]:
GDS_GRAPH_NAME = "sie2025_network"

try:
    ejecutar_query(
        f"CALL gds.graph.drop('{GDS_GRAPH_NAME}', false)"
    )
except Exception:
    pass

ejecutar_query(
    """
    MATCH (n:NetworkActor)
    REMOVE n:NetworkActor
    """
)

# Etiquetar únicamente endpoints conectados.
ejecutar_query(
    """
    MATCH (b:Buyer)-[r:CONTRACTS_WITH]->(s:Supplier)
    WHERE r.dataset = 'SIE_2025'
    SET b:NetworkActor, s:NetworkActor
    """
)

# Proyección no dirigida.
query_project = f"""
CALL gds.graph.project(
    '{GDS_GRAPH_NAME}',
    'NetworkActor',
    {{
        CONTRACTS_WITH: {{
            orientation: 'UNDIRECTED',
            properties: ['frequency', 'total_amount']
        }}
    }}
)
"""

ejecutar_query(query_project)

with driver.session(database=NEO4J_DATABASE) as session:
    graph_info = session.run(
        """
        CALL gds.graph.list($graph_name)
        YIELD graphName, nodeCount, relationshipCount
        RETURN graphName, nodeCount, relationshipCount
        """,
        graph_name=GDS_GRAPH_NAME,
    ).single().data()

print("Proyección GDS creada:")
print(graph_info)
print(
    "Nota: relationshipCount refleja la semántica de la proyección "
    "UNDIRECTED y no debe interpretarse automáticamente como el número "
    "de relaciones Buyer-Supplier originales."
)


In [ ]:
query_wcc_components = """
CALL gds.wcc.stream($graph_name)
YIELD nodeId, componentId
RETURN
    componentId,
    count(*) AS component_size
ORDER BY component_size DESC
"""

with driver.session(database=NEO4J_DATABASE) as session:
    wcc_components_df = pd.DataFrame(
        [
            r.data()
            for r in session.run(
                query_wcc_components,
                graph_name=GDS_GRAPH_NAME,
            )
        ]
    )

n_components = len(wcc_components_df)
giant_component_size = (
    int(wcc_components_df["component_size"].max())
    if not wcc_components_df.empty
    else 0
)
projected_nodes = int(
    wcc_components_df["component_size"].sum()
)
giant_component_share = (
    giant_component_size / projected_nodes
    if projected_nodes
    else 0
)

print("Número de componentes conectados:", n_components)
print("Nodos proyectados:", projected_nodes)
print("Tamaño del componente gigante:", giant_component_size)
print(
    "Proporción en componente gigante:",
    f"{giant_component_share:.2%}",
)

display(wcc_components_df.head(20))


In [ ]:
query_degree_buyers_all = '''
MATCH (b:Buyer)-[:CONTRACTS_WITH]->(s:Supplier)
WHERE b.dataset = 'SIE_2025'
WITH b, count(DISTINCT s) AS degree
RETURN
    b.buyer_id AS actor_id,
    b.name AS actor,
    degree
ORDER BY degree DESC, actor
'''


query_degree_suppliers_all = '''
MATCH (b:Buyer)-[:CONTRACTS_WITH]->(s:Supplier)
WHERE s.dataset = 'SIE_2025'
WITH s, count(DISTINCT b) AS degree
RETURN
    s.supplier_id AS actor_id,
    s.name AS actor,
    degree
ORDER BY degree DESC, actor
'''

with driver.session(database=NEO4J_DATABASE) as session:
    degree_buyers_all = pd.DataFrame(
        [r.data() for r in session.run(query_degree_buyers_all)]
    )

with driver.session(database=NEO4J_DATABASE) as session:
    degree_suppliers_all = pd.DataFrame(
        [r.data() for r in session.run(query_degree_suppliers_all)]
    )

print("Top 20 entidades por número de proveedores distintos")
display(degree_buyers_all.head(20))

print("Top 20 proveedores por número de entidades distintas")
display(degree_suppliers_all.head(20))


In [ ]:
query_strength_buyers_all = '''
MATCH (b:Buyer)-[r:CONTRACTS_WITH]->(:Supplier)
WHERE r.dataset = 'SIE_2025'
WITH
    b,
    count(r) AS degree,
    sum(r.frequency) AS strength_frequency,
    sum(r.total_amount) AS strength_amount
RETURN
    b.buyer_id AS actor_id,
    b.name AS actor,
    degree,
    strength_frequency,
    strength_amount
'''


query_strength_suppliers_all = '''
MATCH (:Buyer)-[r:CONTRACTS_WITH]->(s:Supplier)
WHERE r.dataset = 'SIE_2025'
WITH
    s,
    count(r) AS degree,
    sum(r.frequency) AS strength_frequency,
    sum(r.total_amount) AS strength_amount
RETURN
    s.supplier_id AS actor_id,
    s.name AS actor,
    degree,
    strength_frequency,
    strength_amount
'''

with driver.session(database=NEO4J_DATABASE) as session:
    strength_buyers_all = pd.DataFrame(
        [r.data() for r in session.run(query_strength_buyers_all)]
    )

with driver.session(database=NEO4J_DATABASE) as session:
    strength_suppliers_all = pd.DataFrame(
        [r.data() for r in session.run(query_strength_suppliers_all)]
    )

print("Top 20 entidades por frecuencia acumulada")
display(
    strength_buyers_all
    .sort_values(
        ["strength_frequency", "strength_amount"],
        ascending=[False, False]
    )
    .head(20)
)

print("Top 20 proveedores por frecuencia acumulada")
display(
    strength_suppliers_all
    .sort_values(
        ["strength_frequency", "strength_amount"],
        ascending=[False, False]
    )
    .head(20)
)

print("Top 20 entidades por monto acumulado")
display(
    strength_buyers_all
    .sort_values(
        ["strength_amount", "strength_frequency"],
        ascending=[False, False]
    )
    .head(20)
)

print("Top 20 proveedores por monto acumulado")
display(
    strength_suppliers_all
    .sort_values(
        ["strength_amount", "strength_frequency"],
        ascending=[False, False]
    )
    .head(20)
)


In [ ]:
# Betweenness centrality
#
#

query_betweenness = """
CALL gds.betweenness.stream($graph_name)
YIELD nodeId, score
WITH
    gds.util.asNode(nodeId) AS n,
    score
RETURN
    CASE
        WHEN n:Buyer THEN 'Buyer'
        WHEN n:Supplier THEN 'Supplier'
        ELSE NULL
    END AS actor_type,
    CASE
        WHEN n:Buyer THEN n.buyer_id
        WHEN n:Supplier THEN n.supplier_id
        ELSE NULL
    END AS actor_id,
    n.name AS actor_name,
    score AS betweenness
ORDER BY betweenness DESC
"""

with driver.session(database=NEO4J_DATABASE) as session:
    betweenness_df = pd.DataFrame(
        [
            r.data()
            for r in session.run(
                query_betweenness,
                graph_name=GDS_GRAPH_NAME,
            )
        ]
    )

if betweenness_df.empty:
    raise RuntimeError(
        "gds.betweenness.stream no devolvió resultados."
    )

if betweenness_df["actor_id"].isna().any():
    raise ValueError(
        "Existen nodos proyectados sin actor_id."
    )

if betweenness_df["betweenness"].isna().any():
    raise ValueError(
        "gds.betweenness.stream devolvió valores nulos."
    )

if betweenness_df[
    ["actor_type", "actor_id"]
].duplicated().any():
    raise ValueError(
        "Betweenness contiene actores duplicados."
    )

print(
    "Actores con betweenness calculado:",
    len(betweenness_df)
)
print(
    "Betweenness > 0:",
    int((betweenness_df["betweenness"] > 0).sum())
)
print(
    "Betweenness = 0:",
    int((betweenness_df["betweenness"] == 0).sum())
)

display(
    betweenness_df["betweenness"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("Top 20 actores por betweenness")
display(betweenness_df.head(20))


In [ ]:
query_write_buyer_metrics = '''
MATCH (b:Buyer)-[r:CONTRACTS_WITH]->(s:Supplier)
WHERE r.dataset = 'SIE_2025'
WITH
    b,
    count(DISTINCT s) AS degree,
    sum(r.frequency) AS strength_frequency,
    sum(r.total_amount) AS strength_amount
SET
    b.degree = degree,
    b.strength_frequency = strength_frequency,
    b.strength_amount = strength_amount
RETURN count(b) AS buyers_actualizados
'''

query_write_supplier_metrics = '''
MATCH (b:Buyer)-[r:CONTRACTS_WITH]->(s:Supplier)
WHERE r.dataset = 'SIE_2025'
WITH
    s,
    count(DISTINCT b) AS degree,
    sum(r.frequency) AS strength_frequency,
    sum(r.total_amount) AS strength_amount
SET
    s.degree = degree,
    s.strength_frequency = strength_frequency,
    s.strength_amount = strength_amount
RETURN count(s) AS suppliers_actualizados
'''

with driver.session(database=NEO4J_DATABASE) as session:
    buyers_actualizados = session.run(
        query_write_buyer_metrics
    ).single()["buyers_actualizados"]

with driver.session(database=NEO4J_DATABASE) as session:
    suppliers_actualizados = session.run(
        query_write_supplier_metrics
    ).single()["suppliers_actualizados"]

print("Buyers actualizados:", buyers_actualizados)
print("Suppliers actualizados:", suppliers_actualizados)


In [ ]:
#

try:
    query_write_betweenness = """
    CALL gds.betweenness.write(
        $graph_name,
        {writeProperty: 'betweenness'}
    )
    YIELD nodePropertiesWritten, centralityDistribution
    RETURN nodePropertiesWritten, centralityDistribution
    """

    with driver.session(database=NEO4J_DATABASE) as session:
        betweenness_write_result = (
            session.run(
                query_write_betweenness,
                graph_name=GDS_GRAPH_NAME,
            )
            .single()
            .data()
        )

    print("Persistencia opcional de betweenness:")
    print(betweenness_write_result)

except Exception as exc:
    print(
        "Advertencia: no se pudo persistir betweenness en Neo4j."
    )
    print(
        "La exportación final seguirá usando betweenness_df generado "
        "mediante stream."
    )
    print("Detalle:", exc)


## Export the consolidated network-metrics table


In [ ]:
#

query_actor_metrics = """
MATCH (n)
WHERE
    n.dataset = 'SIE_2025'
    AND (n:Buyer OR n:Supplier)
RETURN
    CASE
        WHEN n:Buyer THEN 'Buyer'
        ELSE 'Supplier'
    END AS actor_type,
    CASE
        WHEN n:Buyer THEN n.buyer_id
        ELSE n.supplier_id
    END AS actor_id,
    n.name AS actor_name,
    coalesce(n.degree, 0) AS degree,
    coalesce(n.strength_frequency, 0) AS strength_frequency,
    coalesce(n.strength_amount, 0.0) AS strength_amount
ORDER BY actor_type, actor_id
"""

with driver.session(database=NEO4J_DATABASE) as session:
    actor_metrics_df = pd.DataFrame(
        [r.data() for r in session.run(query_actor_metrics)]
    )

if actor_metrics_df[
    ["actor_type", "actor_id"]
].duplicated().any():
    raise ValueError(
        "La tabla base de actores contiene claves duplicadas."
    )

actor_metrics_df = actor_metrics_df.merge(
    betweenness_df[
        [
            "actor_type",
            "actor_id",
            "betweenness",
        ]
    ],
    on=[
        "actor_type",
        "actor_id",
    ],
    how="left",
    validate="one_to_one",
)

isolated_missing_mask = (
    actor_metrics_df["betweenness"].isna()
    & actor_metrics_df["degree"].fillna(0).eq(0)
)

n_isolated_filled = int(isolated_missing_mask.sum())

actor_metrics_df.loc[
    isolated_missing_mask,
    "betweenness",
] = 0.0

remaining_missing = int(
    actor_metrics_df["betweenness"].isna().sum()
)

print(
    "Actores aislados asignados a betweenness=0:",
    n_isolated_filled
)
print(
    "Betweenness faltantes después del tratamiento:",
    remaining_missing
)

if remaining_missing != 0:
    problematic = actor_metrics_df.loc[
        actor_metrics_df["betweenness"].isna(),
        [
            "actor_type",
            "actor_id",
            "actor_name",
            "degree",
        ],
    ]

    display(problematic.head(20))

    raise ValueError(
        "Persisten actores conectados sin betweenness. "
        "No se exportará un CSV incompleto."
    )

display(
    actor_metrics_df[
        [
            "degree",
            "strength_frequency",
            "strength_amount",
            "betweenness",
        ]
    ].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

actor_metrics_path = (
    ruta_neo4j
    / "actor_network_metrics_2025.csv"
)

actor_metrics_df.to_csv(
    actor_metrics_path,
    index=False,
    encoding="utf-8-sig",
)

print("Archivo exportado:", actor_metrics_path.resolve())
print("Filas:", len(actor_metrics_df))

validation_metrics_df = pd.read_csv(
    actor_metrics_path,
    dtype={"actor_id": "string"},
)

print(
    "Betweenness no nulos en CSV:",
    int(validation_metrics_df["betweenness"].notna().sum()),
)
print(
    "Betweenness nulos en CSV:",
    int(validation_metrics_df["betweenness"].isna().sum()),
)
print(
    "Betweenness > 0 en CSV:",
    int((validation_metrics_df["betweenness"] > 0).sum()),
)

if validation_metrics_df["betweenness"].isna().any():
    raise ValueError(
        "El CSV exportado contiene betweenness nulos."
    )


In [ ]:
try:
    ejecutar_query(
        f"CALL gds.graph.drop('{GDS_GRAPH_NAME}', false)"
    )
    print("Proyección GDS eliminada.")
except Exception as exc:
    print(
        "No se pudo eliminar la proyección GDS:",
        exc,
    )

try:
    ejecutar_query(
        """
        MATCH (n:NetworkActor)
        REMOVE n:NetworkActor
        """
    )
    print("Etiqueta temporal NetworkActor eliminada.")
except Exception as exc:
    print(
        "No se pudo eliminar NetworkActor:",
        exc,
    )

driver.close()
print("Conexión con Neo4j cerrada.")
